# Standalone Symbolic AI Fall Detection Model (URFD)

**Reference:** Xu et al., *Applied Sciences* 2026, 16(4), 1895 (Sections 3.2 & 3.3.2)  
**Pipeline Inputs:** RTMO 17-keypoints (`(T, 17, 3)`) + Depth-Anything-V2 depth maps (`(T, H, W)`)

### How to use this notebook:
1. Run all cells once right now — the final cell includes a **built-in smoke test** to verify everything works out of the box.
2. Section 7 now reads our preprocessed URFD files directly. Run from the project root (or set `PROJECT_ROOT`), then **Run All**.
3. The notebook automatically runs the same **2 scene-disjoint folds as the ST-GCN** (`split_scene_v2.json`), prints both **Frame-Level** and **Event-Level** metrics, and saves per-video symbolic outputs (`symbolic_predictions.npz`) ready for later **Neuro-Symbolic Fusion** with the ST-GCN stream.

## 1. Imports & Configuration (`SymbolicConfig`)

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import glob
import os
import numpy as np


@dataclass
class SymbolicConfig:
    fps: float = 30.0
    conf_threshold: float = 0.25           # RTMO keypoint confidence threshold
    depth_patch_radius: int = 1            # 3x3 median patch around joint to avoid depth noise
    torso_hip_to_neck_ratio: float = 0.35  # Vertical offset when shoulders disappear on floor

    # Physical thresholds (auto-tuned on training fold via fit_thresholds())
    height_threshold: float = 0.35         # tau_h: floor-relative neck height threshold
    velocity_threshold: float = 0.45       # tau_v: downward vertical velocity threshold
    descent_delta_sec: float = 0.20        # Delta_t for velocity computation (~6 frames @ 30fps)

    # Temporal smoothing & URFD post-fall lying behavior
    sym_smooth_sec: float = 0.25           # Eq. (10): sliding-window majority smoothing
    mode: str = "urfd_state_aware"         # "strict_and" (Paper Eq. 8&9) or "urfd_state_aware"
    lying_aspect_threshold: float = 0.35   # Max normalized torso vertical span when lying flat

## 2. Input Data Contract (`SymbolicVideoInput`)

In [ ]:
@dataclass
class SymbolicVideoInput:
    video_id: str
    scene_id: str                                # Room/scene ID for leak-free 2-fold split
    keypoints: np.ndarray                        # Shape: (T, 17, 3) -> [x_px, y_px, confidence]
    depth_maps: Optional[np.ndarray] = None      # Shape: (T, H_d, W_d) from Depth-Anything-V2
    joint_depths: Optional[np.ndarray] = None    # Optional pre-sampled joint depths: (T, 17)
    image_size: Optional[Tuple[int, int]] = None # Optional (H_rgb, W_rgb) if depth_maps was resized
    labels: Optional[np.ndarray] = None          # Ground-truth frame labels: (T,) in {0, 1}

## 3. Pose & Depth Feature Extractor (`PoseDepthExtractor`)
Extracts Neck (with **Hip fallback** when shoulders are occluded on the floor), Hip, and Ankle `(u, v, depth)` triplets per frame.

In [ ]:
class PoseDepthExtractor:
    """
    Extracts Neck (with Hip fallback when occluded on the floor), Hip, and Ankle
    triplets (u, v, depth) per frame.
    """
    NOSE = 0
    L_SHOULDER, R_SHOULDER = 5, 6
    L_HIP, R_HIP = 11, 12
    L_ANKLE, R_ANKLE = 15, 16

    def __init__(self, cfg: SymbolicConfig):
        self.cfg = cfg

    def _sample_depth(
        self,
        depth_map: np.ndarray,
        u: float,
        v: float,
        image_size: Optional[Tuple[int, int]] = None,
    ) -> float:
        H_d, W_d = depth_map.shape
        if image_size is not None:
            H_rgb, W_rgb = image_size
            u = u * (W_d / max(W_rgb, 1))
            v = v * (H_d / max(H_rgb, 1))

        x = int(np.clip(round(u), 0, W_d - 1))
        y = int(np.clip(round(v), 0, H_d - 1))
        r = self.cfg.depth_patch_radius
        patch = depth_map[max(0, y - r):min(H_d, y + r + 1), max(0, x - r):min(W_d, x + r + 1)]
        return float(np.median(patch))

    def _get_midpoint(
        self, kpts_t: np.ndarray, idx1: int, idx2: int
    ) -> Tuple[Optional[np.ndarray], float]:
        p1, p2 = kpts_t[idx1], kpts_t[idx2]
        c1, c2 = p1[2], p2[2]
        th = self.cfg.conf_threshold
        if c1 >= th and c2 >= th:
            return 0.5 * (p1[:2] + p2[:2]), float(min(c1, c2))
        if c1 >= th:
            return p1[:2].copy(), float(c1)
        if c2 >= th:
            return p2[:2].copy(), float(c2)
        return None, 0.0

    def extract(self, sample: SymbolicVideoInput) -> Dict[str, np.ndarray]:
        T = sample.keypoints.shape[0]
        neck_uvd = np.full((T, 3), np.nan, dtype=np.float32)
        hip_uvd = np.full((T, 3), np.nan, dtype=np.float32)
        ankle_uvd = np.full((T, 3), np.nan, dtype=np.float32)
        valid_mask = np.zeros(T, dtype=bool)

        # Estimate standing torso pixel length in this video for hip-to-neck fallback
        torso_lengths = []
        for t in range(T):
            s_pt, _ = self._get_midpoint(sample.keypoints[t], self.L_SHOULDER, self.R_SHOULDER)
            h_pt, _ = self._get_midpoint(sample.keypoints[t], self.L_HIP, self.R_HIP)
            if s_pt is not None and h_pt is not None:
                torso_lengths.append(np.linalg.norm(s_pt - h_pt))
        median_torso_px = float(np.median(torso_lengths)) if torso_lengths else 60.0

        for t in range(T):
            kpts_t = sample.keypoints[t]
            s_pt, _ = self._get_midpoint(kpts_t, self.L_SHOULDER, self.R_SHOULDER)
            h_pt, _ = self._get_midpoint(kpts_t, self.L_HIP, self.R_HIP)
            a_pt, _ = self._get_midpoint(kpts_t, self.L_ANKLE, self.R_ANKLE)

            # Neck extraction with Hip fallback when shoulders disappear on floor
            if s_pt is not None:
                n_pt = s_pt
            elif h_pt is not None:
                n_pt = np.array(
                    [h_pt[0], h_pt[1] - self.cfg.torso_hip_to_neck_ratio * median_torso_px],
                    dtype=np.float32,
                )
            elif kpts_t[self.NOSE, 2] >= self.cfg.conf_threshold:
                n_pt = kpts_t[self.NOSE, :2].copy()
            else:
                continue

            valid_mask[t] = True

            def get_depth(pt: np.ndarray, pair: Tuple[int, int]) -> float:
                if sample.joint_depths is not None:
                    return float(np.nanmean(sample.joint_depths[t, list(pair)]))
                if sample.depth_maps is None:
                    raise ValueError("Provide either `depth_maps` or `joint_depths`.")
                return self._sample_depth(sample.depth_maps[t], pt[0], pt[1], sample.image_size)

            neck_uvd[t] = [n_pt[0], n_pt[1], get_depth(n_pt, (self.L_SHOULDER, self.R_SHOULDER))]
            if h_pt is not None:
                hip_uvd[t] = [h_pt[0], h_pt[1], get_depth(h_pt, (self.L_HIP, self.R_HIP))]
            if a_pt is not None:
                ankle_uvd[t] = [a_pt[0], a_pt[1], get_depth(a_pt, (self.L_ANKLE, self.R_ANKLE))]

        # Linearly interpolate short missing neck gaps
        for col in range(3):
            series = neck_uvd[:, col]
            nans = np.isnan(series)
            if np.any(nans) and not np.all(nans):
                idx = np.arange(T)
                series[nans] = np.interp(idx[nans], idx[~nans], series[~nans])
                neck_uvd[:, col] = series

        return {
            "neck_uvd": neck_uvd,
            "hip_uvd": hip_uvd,
            "ankle_uvd": ankle_uvd,
            "valid_mask": valid_mask,
        }

## 4. Floor-Relative Height Estimator (`FloorHeightEstimator`)
Recovers vertical neck height $h_{\text{neck}}(t)$ relative to the floor plane using either 3D room markers (Least Squares) or automatic monocular floor/body-scale calibration for URFD.

In [ ]:
class FloorHeightEstimator:
    """
    Computes vertical neck height h_neck(t) relative to the floor plane:
      - If 3D room calibration points are supplied: uses Paper Least-Squares T (3x4).
      - Otherwise (URFD default): self-calibrates floor reference & body scale from kinematics.
    """
    def __init__(self, transform_3x4: Optional[np.ndarray] = None, floor_z: float = 0.0):
        self.T = transform_3x4
        self.floor_z = floor_z

    @classmethod
    def fit_from_3d_markers(
        cls, uvd_points: np.ndarray, xyz_world: np.ndarray, floor_z: float = 0.0
    ) -> "FloorHeightEstimator":
        N = uvd_points.shape[0]
        X_aug = np.hstack([uvd_points, np.ones((N, 1))])
        W, _, _, _ = np.linalg.lstsq(X_aug, xyz_world, rcond=None)
        return cls(transform_3x4=W.T, floor_z=floor_z)

    def compute_height(self, triplets: Dict[str, np.ndarray]) -> Tuple[np.ndarray, np.ndarray]:
        neck = triplets["neck_uvd"]
        hip = triplets["hip_uvd"]
        ankle = triplets["ankle_uvd"]
        T = neck.shape[0]

        if self.T is not None:
            ones = np.ones((T, 1), dtype=np.float32)
            neck_xyz = (self.T @ np.hstack([neck, ones]).T).T
            h_neck = neck_xyz[:, 2] - self.floor_z
            if not np.all(np.isnan(hip)):
                hip_xyz = (self.T @ np.hstack([hip, ones]).T).T
                torso_span = np.abs(neck_xyz[:, 2] - hip_xyz[:, 2])
            else:
                torso_span = np.full(T, np.nan, dtype=np.float32)
            return h_neck, torso_span

        # Monocular self-calibration for URFD (Depth-Anything-V2 outputs relative disparity)
        v_neck = neck[:, 1]
        d_neck = neck[:, 2]
        d_med = np.nanmedian(d_neck) if np.nanmedian(d_neck) > 1e-6 else 1.0
        disparity_scale = np.clip(d_neck / d_med, 0.5, 2.0)

        valid_both = ~np.isnan(ankle[:, 1]) & ~np.isnan(v_neck)
        if np.sum(valid_both) >= 5:
            body_spans = (ankle[valid_both, 1] - v_neck[valid_both]) / disparity_scale[valid_both]
            standing_span = float(np.percentile(body_spans, 85))
            v_floor_ref = float(np.percentile(ankle[valid_both, 1], 90))
        else:
            v_top = float(np.percentile(v_neck, 10))
            v_bot = float(np.percentile(v_neck, 95))
            standing_span = max((v_bot - v_top) / 0.85, 80.0)
            v_floor_ref = v_top + standing_span

        standing_span = max(standing_span, 40.0)

        # Normalized floor height: ~1.0 when standing, < 0.3 when near floor
        h_neck = np.clip((v_floor_ref - v_neck) / (standing_span * disparity_scale), -0.2, 1.5)

        torso_span = np.full(T, np.nan, dtype=np.float32)
        valid_hip = ~np.isnan(hip[:, 1])
        torso_span[valid_hip] = np.abs(hip[valid_hip, 1] - v_neck[valid_hip]) / (
            standing_span * disparity_scale[valid_hip]
        )

        return h_neck, torso_span

## 5. Symbolic AI Fall Detector (`SymbolicFallDetector`)
Applies physical constraints:
1. **Floor Proximity:** $h_{\text{neck}}(t) < \tau_h$
2. **Rapid Vertical Descent:** $v_z(t) > \tau_v$
3. **Temporal Consistency & URFD Lying State Latch**
4. **Leak-Free Threshold Fitting (`fit_thresholds`)** on training scenes only maximizing MCC.

In [ ]:
class SymbolicFallDetector:
    """
    Pure Symbolic AI Fall Detector using physical rules:
      - Rule 1 (Eq. 8): Floor proximity -> h_neck(t) < tau_h
      - Rule 2 (Eq. 9): Rapid vertical descent -> v_z(t) > tau_v
      - Rule 3 (Eq. 10): Temporal majority consistency + optional URFD lying-state persistence
    """
    def __init__(
        self,
        cfg: Optional[SymbolicConfig] = None,
        height_estimator: Optional[FloorHeightEstimator] = None,
    ):
        self.cfg = cfg or SymbolicConfig()
        self.extractor = PoseDepthExtractor(self.cfg)
        self.height_estimator = height_estimator or FloorHeightEstimator()

    @staticmethod
    def _majority_smooth(seq: np.ndarray, win_frames: int) -> np.ndarray:
        if win_frames <= 1:
            return seq.astype(int)
        kernel = np.ones(win_frames, dtype=int)
        counts = np.convolve(seq.astype(int), kernel, mode="same")
        return (counts >= (win_frames // 2 + 1)).astype(int)

    def predict_video(self, sample: SymbolicVideoInput) -> Dict[str, np.ndarray]:
        triplets = self.extractor.extract(sample)
        h_neck, torso_span = self.height_estimator.compute_height(triplets)
        T = len(h_neck)

        # Vertical downward velocity: v_z(t) = (h(t - dt) - h(t)) / dt
        lag = max(1, int(round(self.cfg.descent_delta_sec * self.cfg.fps)))
        dt = lag / self.cfg.fps
        v_z = np.zeros(T, dtype=np.float32)
        if T > lag:
            v_z[lag:] = (h_neck[:-lag] - h_neck[lag:]) / dt

        cond_floor = h_neck < self.cfg.height_threshold      # Eq. (8)
        cond_descent = v_z > self.cfg.velocity_threshold     # Eq. (9)

        if self.cfg.mode == "strict_and":
            # Exact Paper Rule: both conditions hold simultaneously
            raw_sym = cond_floor & cond_descent
        else:
            # URFD-aware rule: triggers on descent + floor proximity, and stays active
            # while the subject remains lying horizontally on the floor.
            raw_sym = np.zeros(T, dtype=bool)
            in_fall = False
            lookback = int(round(1.5 * self.cfg.fps))

            for t in range(T):
                recent_drop = np.any(cond_descent[max(0, t - lookback):t + 1])
                is_horizontal = (
                    True if np.isnan(torso_span[t]) else (torso_span[t] < self.cfg.lying_aspect_threshold)
                )
                if cond_floor[t] and recent_drop:
                    in_fall = True
                elif in_fall and cond_floor[t] and is_horizontal:
                    in_fall = True
                else:
                    in_fall = False
                raw_sym[t] = in_fall

        smooth_win = max(1, int(round(self.cfg.sym_smooth_sec * self.cfg.fps)))
        y_sym = self._majority_smooth(raw_sym, smooth_win)

        return {
            "y_sym": y_sym,                    # Binary frame prediction: (T,) in {0, 1}
            "cond_floor": cond_floor.astype(int),
            "cond_descent": cond_descent.astype(int),
            "h_neck": h_neck,
            "v_z": v_z,
        }

    def fit_thresholds(
        self,
        train_samples: List[SymbolicVideoInput],
        height_grid: np.ndarray = np.linspace(0.20, 0.50, 13),
        velocity_grid: np.ndarray = np.linspace(0.25, 0.85, 13),
    ) -> Tuple[float, float, float]:
        """
        Finds optimal (tau_h, tau_v) on TRAIN FOLD ONLY maximizing MCC (no test-scene leakage).
        """
        best_mcc = -1.0
        best_h, best_v = self.cfg.height_threshold, self.cfg.velocity_threshold

        for th in height_grid:
            for tv in velocity_grid:
                self.cfg.height_threshold = float(th)
                self.cfg.velocity_threshold = float(tv)
                y_trues, y_preds = [], []
                for s in train_samples:
                    if s.labels is None:
                        raise ValueError(f"Missing ground-truth labels for train video: {s.video_id}")
                    y_trues.append(s.labels)
                    y_preds.append(self.predict_video(s)["y_sym"])
                mcc = compute_frame_metrics(np.concatenate(y_trues), np.concatenate(y_preds))["MCC"]
                if mcc > best_mcc:
                    best_mcc = mcc
                    best_h, best_v = float(th), float(tv)

        self.cfg.height_threshold = best_h
        self.cfg.velocity_threshold = best_v
        return best_h, best_v, best_mcc

## 6. Evaluation Metrics (Frame-Level & Event-Level)

In [ ]:
def compute_frame_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    y_true = y_true.astype(int)
    y_pred = y_pred.astype(int)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))

    acc = (tp + tn) / max(tp + tn + fp + fn, 1)
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    f1 = (2 * prec * rec) / max(prec + rec, 1e-8)
    denom = np.sqrt(float((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)))
    mcc = ((tp * tn) - (fp * fn)) / denom if denom > 0 else 0.0

    return {
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1": round(f1, 4),
        "MCC": round(mcc, 4),
    }


def compute_event_metrics(
    samples: List[SymbolicVideoInput], predictions: List[np.ndarray], fps: float = 30.0, min_frames: int = 3
) -> Dict[str, object]:
    fall_vids, falls_caught = 0, 0
    adl_vids, clean_adl = 0, 0
    fa_events, adl_minutes = 0, 0.0

    for s, pred in zip(samples, predictions):
        is_fall = (s.labels is not None and np.any(s.labels == 1)) or ("fall" in s.video_id.lower())
        padded = np.concatenate([[0], (pred > 0).astype(int), [0]])
        diffs = np.diff(padded)
        starts = np.where(diffs == 1)[0]
        ends = np.where(diffs == -1)[0]
        events = sum(1 for st, en in zip(starts, ends) if (en - st) >= min_frames)

        if is_fall:
            fall_vids += 1
            if events >= 1:
                falls_caught += 1
        else:
            adl_vids += 1
            if events == 0:
                clean_adl += 1
            fa_events += events
            adl_minutes += (len(pred) / fps) / 60.0

    return {
        "Falls_Caught": f"{falls_caught}/{fall_vids}",
        "Clean_ADL_Videos": f"{clean_adl}/{adl_vids}",
        "False_Alarms_Per_Min": round(fa_events / max(adl_minutes, 1e-6), 2),
    }

## 7. Data Adapter for our URFD files + our scene folds
Reads the preprocessed files we already have (`processed_data/cfg_d31bfed741ab/urfd_npy`) and the saved split `splits/split_scene_v2.json`, so the symbolic stream is evaluated on exactly the same test videos as the ST-GCN.

- keypoints: `<video>_raw_coco17.npy`, shape (T, 17, 3), pixels on a 640x480 frame
- depth: `<video>_neck_depth.npy` (Depth-Anything-V2 at the neck, NaN where no anchor). Only the neck depth is used by the monocular height estimator, so it is passed as `joint_depths`
- labels: `<video>_frame_labels.npy`, 0/1, the same labels the ST-GCN uses

Protocol: thresholds are fitted on each fold's **train** videos only, the rule (`strict_and` or `urfd_state_aware`) is chosen by mean **validation** MCC, and only then is the test fold scored. Event metrics use the ST-GCN definitions so the two streams can be compared.

In [ ]:
import json, warnings
PROJECT_ROOT = os.environ.get("PROJECT_ROOT", ".")
NPY_DIR = os.path.join(PROJECT_ROOT, "processed_data", "cfg_d31bfed741ab", "urfd_npy")
SPLIT_FILE = os.path.join(PROJECT_ROOT, "splits", "split_scene_v2.json")
HEIGHT_GRID = np.linspace(0.10, 0.80, 15)    # the original grids put the best value on the edge
VELOCITY_GRID = np.linspace(0.25, 2.0, 15)


def load_urfd_data(npy_dir: str = NPY_DIR, split_file: str = SPLIT_FILE) -> Dict[str, SymbolicVideoInput]:
    split = json.load(open(split_file))
    data = {}
    for vid, scene in sorted(split["group_of"].items()):
        kpts = np.load(os.path.join(npy_dir, f"{vid}_raw_coco17.npy")).astype(np.float32)
        neck_d = np.load(os.path.join(npy_dir, f"{vid}_neck_depth.npy")).astype(np.float32)
        labels = np.load(os.path.join(npy_dir, f"{vid}_frame_labels.npy")).astype(int)
        data[vid] = SymbolicVideoInput(vid, scene, kpts, joint_depths=np.repeat(neck_d[:, None], 17, axis=1),
                                       image_size=(480, 640), labels=labels)
    return data


def stgcn_event_metrics(samples, preds, fps=30.0):
    """Same definitions as final_test_report.py: a fall counts if any frame from the onset on is positive;
    an ADL video is clean if no frame is positive; false-alarm episodes per minute of ADL video."""
    det = falls = clean = adl = eps = frames = 0
    for s, p in zip(samples, preds):
        p = np.asarray(p).astype(bool)
        if s.video_id.startswith("fall"):
            falls += 1; on = int(np.argmax(s.labels)); det += int(p[on:].any())
        else:
            adl += 1; clean += int(not p.any()); frames += len(p)
            eps += int(p[0]) + int((np.diff(p.astype(int)) == 1).sum())
    return {"Falls_Detected": f"{det}/{falls}", "ADL_No_False_Alarm": f"{clean}/{adl}",
            "False_Alarms_Per_Min": round(eps / max(frames / fps / 60, 1e-6), 2)}


def run_symbolic_scene_folds(data, split_file=SPLIT_FILE, modes=("strict_and", "urfd_state_aware")):
    warnings.filterwarnings("ignore", category=RuntimeWarning)
    folds = json.load(open(split_file))["folds"]
    fitted, val_mcc = {}, {m: [] for m in modes}
    for m in modes:
        for f in folds:
            model = SymbolicFallDetector(cfg=SymbolicConfig(fps=30.0, mode=m))
            h, v, tr = model.fit_thresholds([data[x] for x in f["train"]], HEIGHT_GRID, VELOCITY_GRID)
            va = [data[x] for x in f["val"]]
            vm = compute_frame_metrics(np.concatenate([s.labels for s in va]),
                                       np.concatenate([model.predict_video(s)["y_sym"] for s in va]))["MCC"]
            fitted[(m, f["fold"])] = model; val_mcc[m].append(vm)
            print(f"{m:17s} fold {f['fold']}: tau_h={h:.3f} tau_v={v:.3f} | train MCC={tr:.3f} | val MCC={vm:.3f}")
    best = max(modes, key=lambda m: np.mean(val_mcc[m]))
    print(f"\nRule chosen on validation: {best}")

    outputs, samples, preds = {}, [], []
    for f in folds:
        model = fitted[(best, f["fold"])]
        for vid in f["test"]:
            out = model.predict_video(data[vid]); out["fold"] = f["fold"]
            outputs[vid] = out; samples.append(data[vid]); preds.append(out["y_sym"])
    print("=== SYMBOLIC MODEL on the ST-GCN test folds (Frame-Level) ===")
    print(compute_frame_metrics(np.concatenate([s.labels for s in samples]), np.concatenate(preds)))
    print("=== SYMBOLIC MODEL (Event-Level, ST-GCN definitions) ===")
    print(stgcn_event_metrics(samples, preds))
    return outputs

## 8. Run Evaluation (or Synthetic Smoke Test)

In [ ]:
data = load_urfd_data()
print(f"Loaded {len(data)} videos")
symbolic_outputs = run_symbolic_scene_folds(data)

# Per-frame outputs for the fusion with the ST-GCN ensemble (same frame order as the *_frame_labels.npy files)
save_dict = {}
for vid, out in symbolic_outputs.items():
    for key in ("y_sym", "cond_floor", "cond_descent", "h_neck", "v_z"):
        save_dict[f"{vid}__{key}"] = out[key]
np.savez(os.path.join(PROJECT_ROOT, "symbolic_predictions.npz"), **save_dict)
print("Saved symbolic_predictions.npz")